In [ ]:
import sys
sys.path.append("../..")      # goes up two folders to the project root, where db_config.py lives
import pandas as pd
from db_config import run_query

orgc = run_query("SELECT * FROM wosis_orgc WHERE country_name = 'Ghana';")
orgc[['profile_id', 'layer_id', 'upper_depth', 'lower_depth', 'value_avg']].head(10)
orgc.info()
orgc.describe()

In [ ]:
high = orgc[orgc["value_avg"] > 200]
print(len(high))
high[["profile_id", "upper_depth", "lower_depth", "value_avg", "organic_surface", "dataset_id"]].head(10)

In [ ]:
# Step 3a: Count which survey they came from for the whole dataset(B2)

print(orgc["dataset_id"].value_counts())

In [ ]:
# The first counts True and False among the 144 high layers. If high is gone after a kernel restart, recreate it with high = orgc[orgc["value_avg"] > 200].
# The second lists the licences in your Ghana table, which also completes your README.

print(high["organic_surface"].value_counts())
print(orgc["licence"].value_counts())

In [ ]:
# Step 3b: Count which survey they came from for the whole dataset(B2)
print(high['dataset_id'].value_counts())

In [ ]:
layers_per_profile = orgc.groupby("profile_id").size()
print(layers_per_profile.sum())
print(layers_per_profile.describe())

In [ ]:
orgc.groupby("profile_id").size()



In [ ]:
print(orgc["organic_surface"].value_counts())

In [ ]:
orgc[["value", "value_avg"]].head(5)

In [ ]:
orgc.nunique()

# Day 81 Notes — Explained Simply (WoSIS, Ghana, Organic Carbon)

## 1. Rows and Columns

The table has **1,756 rows** and **19 columns**.

- **One row** = one soil layer (a slice of soil at a certain depth).
- **One column** = one piece of information about that layer.

The rows come from **318 different soil profiles** (soil pits).  
On average, each profile has about **5.5 layers**.

That is why the same `profile_id` appears many times — once for each layer in that profile.

> **How do I know there are 318 profiles?**  
> I counted how many different `profile_id` values exist.  
> In code: `orgc['profile_id'].nunique()` → 318.  
> SQL:
> ```sql
> SELECT COUNT(DISTINCT profile_id)
> FROM wosis_orgc
> WHERE country_name = 'Ghana';
> ```
> The 1,756 rows are layers, not profiles.

---

## 2. Important Variables

| Column | Simple meaning |
|---|---|
| `value_avg` | The organic carbon number for that layer. Unit = g/kg. |
| `upper_depth` | Top of the layer (cm from surface). |
| `lower_depth` | Bottom of the layer (cm from surface). |
| `latitude`, `longitude` | Where the sample was taken. |
| `date` | When it was sampled. Stored as text, not a real date. |
| `dataset_id` | Which survey collected it. Like a source tag. |
| `organic_surface` | True/False — is this an organic layer on top of mineral soil? **Constant: `False` in all 1,756 rows.** |
| `licence` | The rules for using the data (see Licence Values below). |
| `profile_id`, `layer_id` | ID numbers. Not measurements. Just labels. |

**Key point:** `value_avg` is the carbon number. Everything else helps to use it or check it.

### Licence Values

| Licence | Rows | Share |
|---------|------|-------|
| CC BY 4.0 | 955 | 54.4% |
| CC BY-NC 3.0 | 722 | 41.1% |
| CC BY 3.0 | 79 | 4.5% |

---

## 3. Missing Values

- `layer_name` is missing in **67% of rows**. That is fine because I do not need it.
- `date` has hidden gaps:
  - **74 rows** have `????-??-??` — completely unknown.
  - Some have partial dates like `2009-??-??` — year known, month/day unknown.

So `date` needs cleaning before it can be used.

---

## 4. Suspicious Values

This is the most important part.

The carbon values:

| Statistic | Value | In percent |
|---|---|---|
| Median | 35.8 g/kg | 3.58% |
| 75th percentile | 108.5 g/kg | 10.85% |
| Maximum | 638.7 g/kg | 63.87% |
| Mean | 70.4 g/kg | 7.04% |

**What this tells me:**

- The **mean is double the median**. That means a few very large numbers are pulling the average up. The data is **skewed** — not a nice bell curve.
- **144 layers (8.2%)** are above **200 g/kg (20%)**. I chose 20% as my screening line.
- **All 144** come from **one survey**: `GH-GhaSP`. That survey has 935 rows (53% of my data). Inside it, 15% of rows are above 20%.
- The **other four surveys have zero** suspicious values.

**Strange details:**

- I checked `organic_surface` and it is `False` for all 1,756 rows, so it is a constant column and cannot explain the 144 high-carbon layers.
- Some high values are **deep**: 366 g/kg at 73–103 cm. That is **unusual** — carbon normally drops with depth. Buried organic layers exist, so I should not call it unlikely.
- Two layers in profile 1169146 **overlap**. That is a data quality problem.

**What I did:** I identified the 144 layers. I did **not** delete them.  
The plan is to flag them on Days 87–89.  
That is correct because the cause is not yet known. Possible causes:

- Different lab method
- Wrong quantity recorded as organic carbon
- Real organic-rich soil (unusual at that depth)

---

## 5. Useful Target Variable

`value_avg` — organic carbon in g/kg.  
This is what my model will try to predict.

---

## 6. Possible Predictor Variables

| Predictor | Why it matters |
|---|---|
| `upper_depth`, `lower_depth` | Carbon usually drops with depth. Shallow layers usually have more. |
| `latitude`, `longitude` | Climate and soil may change across Ghana. |
| `date` (year) | Carbon may change over time. |
| pH, clay, sand (after Day 83 join) | Clay may hold carbon. pH may affect microbes. |
| `dataset_id` | **Not** a predictor. It tells me who recorded it, not why carbon varies. It will be used to check quality. |

### Columns I leave out of the predictor list

| Group | Columns | Reason |
|---|---|---|
| Identifiers | `profile_id`, `layer_id`, `profile_code` | Labels for joining and grouping, not soil properties. Including them risks memorising profiles instead of learning depth/location effects. |
| Constants | `organic_surface`, `country_name`, `region`, `continent` | Each has only one unique value in the data, so zero variance and no predictive information. |
| Metadata / source | `dataset_id`, `licence`, `layer_name`, `method_options`, `positional_uncertainty` | They describe who collected the data, legal terms, method details, or are mostly missing (`layer_name` is 67% missing). They do not cause carbon variation. `dataset_id` is for quality checks, not prediction. |
| Raw date | `date` (raw text) | 74 rows are completely unknown and some are partial. Only a cleaned `year` might be tested, and even then it is survey timing, not a soil-forming factor. |
| Target leak | `value` | It is the raw measurement that `value_avg` is derived from, so it would leak the target. |

---

## 7. Honest Summary

> The table holds 1,756 soil layers from 318 profiles across Ghana, with `value_avg` as organic carbon in g/kg.  
> The biggest problems are 144 layers (8.2%) above 200 g/kg — all from `GH-GhaSP`, where 15% of its 935 rows are high — plus 67% missing `layer_name` and 74 rows with completely unknown dates (`????-??-??`); I identified the high carbon values but did not delete them; flagging is planned for Days 87–89.  
> This dataset can support modelling organic carbon concentration by depth, location, and later pH/clay/sand, but it cannot support carbon stock because bulk density is not loaded, and it cannot support recent conditions because nothing in the table is newer than 2009.

---

## The Big Picture

I am doing **data quality control**.  
Before training any AI model, I must:

1. Know what each column means.
2. Find missing values.
3. Find suspicious values.
4. Decide: delete, fix, or flag?
5. Choose my target and predictors.

Suspicious values were identified.  
Investigating why `GH-GhaSP` has so many high values belongs in cleaning (Days 87–88), not my immediate next step.  
My immediate next step is Day 82, the data dictionary.

---

## Quick Recap

- **1,756 rows** = soil layers.
- **318 profiles** = unique soil pits.
- **`value_avg`** = organic carbon (g/kg).
- **Missing:** `layer_name` (67%), `date` (some unknown; 74 rows `????-??-??`).
- **Suspicious:** 144 layers > 200 g/kg, all from `GH-GhaSP`. Identified, not deleted. Flagging planned for Days 87–89.
- **Next:** Day 82 data dictionary. **Later:** cleaning Days 87–88: investigate `GH-GhaSP` and clean `date`.